# BOLD Mortality Prediction: First Pass

Predicting whether an ICU patient died during their hospital admission, using only labs and
vitals available at a single point in time, and explaining *which* markers drive that
prediction rather than just producing a score.

Kept simple on purpose: logistic regression, then a Random Forest with SHAP, then a basic
check on whether the top markers still hold up once we account for how sick the patient
already was.

The transformer model and proper causal inference come later. Not here.

## Step 0: Data access

Real BOLD is a credentialed PhysioNet dataset and certification is still in progress. Until
`bold_dataset.csv` is in this folder, the notebook runs on a synthetic stand-in with the same
column names. That is enough to build the pipeline, not enough to trust any result.

Switching over is one line.

In [1]:
# Swap to "bold_dataset.csv" when the real data arrives.
DATA_PATH = "synthetic_bold_dataset.csv"
IS_SYNTHETIC = DATA_PATH != "bold_dataset.csv"

# Warn when the data is synthetic.
print("=" * 70)
if IS_SYNTHETIC:
    print("  SYNTHETIC DATA. Every number below is made up.")
    print("  This shows the pipeline runs. It is not a finding.")
else:
    print("  REAL BOLD DATA. Results below are genuine.")
print("=" * 70)

  SYNTHETIC DATA. Every number below is made up.
  This shows the pipeline runs. It is not a finding.


## Step 1: Load and look

Look at the data before touching it. Check how many patients, how many columns, and how much of each column is filled in.

Some columns are mostly full because those tests get run on nearly every patient. Others have more gaps because they only get ordered when a doctor already suspects 
something specific. That's expected, not a problem.

Nothing gets dropped or changed yet. This step is just looking.

In [2]:
import pandas as pd

# Don't truncate long tables.
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

df = pd.read_csv(DATA_PATH)

print(f"Loaded:   {DATA_PATH}")
print(f"Patients: {df.shape[0]:,}")
print(f"Columns:  {df.shape[1]}")

Loaded:   synthetic_bold_dataset.csv
Patients: 2,000
Columns:  71


In [3]:
# Count columns by type, then list the non-numeric ones.
print("Column types:")
print(df.dtypes.value_counts())
print()

text_cols = df.select_dtypes(exclude="number").columns.tolist()
print("Text columns:", ", ".join(text_cols) if text_cols else "none")

Column types:
float64    51
int64      18
str         2
Name: count, dtype: int64

Text columns: source_db, race_ethnicity


In [4]:
# Missing values per column, worst first.
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
}).sort_values("pct_missing", ascending=False)

print("Missing data per column (worst first):")
print(missing.to_string())

Missing data per column (worst first):
                                 n_missing  pct_missing
others_ck_mb                          1109         55.4
others_ck_cpk                          902         45.1
hfp_bilirubin_direct                   806         40.3
others_ld_ldh                          780         39.0
coag_fibrinogen                        687         34.4
hfp_albumin                            669         33.4
hfp_alp                                653         32.6
hfp_bilirubin_total                    615         30.8
hfp_alt                                611         30.6
hfp_ast                                594         29.7
bmp_lactate                            476         23.8
coag_ptt                               413         20.6
coag_inr                               293         14.6
coag_pt                                286         14.3
cbc_rdw                                198          9.9
bmp_bun                                  0          0.0
cbc_rbc  

In [ ]:
# Bucket columns by how much is missing.
pct = missing["pct_missing"]
buckets = {
    "Complete (0%)":       (pct == 0).sum(),
    "Usable (1-30%)":      ((pct > 0) & (pct <= 30)).sum(),
    "Patchy (31-60%)":     ((pct > 30) & (pct <= 60)).sum(),
    "Mostly empty (>60%)": (pct > 60).sum(),
}
for name, n in buckets.items():
    print(f"{name:<22}{n:>3} columns")
print()

Complete (0%)          56 columns
Usable (1-30%)          6 columns
Patchy (31-60%)         9 columns
Mostly empty (>60%)     0 columns

Nothing is over the 60% line, so the Step 4 drop rule won't remove anything here.


## Step 2: Define the label

This is what we're actually predicting: `in_hospital_mortality`. A value of 1 means the patient died, 0 means they survived. BOLD already provides this label directly, nothing to build.

The important part is how rare deaths are, because that changes everything else we do later:

1. **Accuracy won't tell us anything useful.** A model that just guesses "survived" every time would still be right about 85% of the time, while catching zero deaths. That's why we use AUROC and AUPRC instead (Step 5), not plain accuracy.
2. **The model needs to be told deaths matter more.** Otherwise it will happily ignore the rare cases to keep its overall error low. 
3. **The train/test split has to keep the same death rate in both halves.** Otherwise one half could end up with way more or fewer deaths than the other by chance (Step 4).

In [6]:
LABEL = "in_hospital_mortality"

# Fail early if the label has gaps or isn't 0/1.
assert LABEL in df.columns, f"'{LABEL}' not found"
assert df[LABEL].isna().sum() == 0, "Label has missing values"
assert set(df[LABEL].unique()) <= {0, 1}, f"Expected 0/1, got {sorted(df[LABEL].unique())}"

print(f"Label: {LABEL}")
print("No gaps, values are 0/1 only.")

Label: in_hospital_mortality
No gaps, values are 0/1 only.


In [7]:
# Count deaths against survivors.
counts = df[LABEL].value_counts()
n_survived = int(counts.get(0, 0))
n_died = int(counts.get(1, 0))
pct_died = n_died / len(df) * 100

print(f"Survived: {n_survived:>6,}  ({100 - pct_died:.1f}%)")
print(f"Died:     {n_died:>6,}  ({pct_died:.1f}%)")
print(f"Total:    {len(df):>6,}")
print()
print(f"Roughly {n_survived / n_died:.1f} survivors for every death.")

Survived:  1,695  (84.8%)
Died:        305  (15.2%)
Total:     2,000

Roughly 5.6 survivors for every death.


In [8]:
# Sanity checks on the death rate.
print(f"An 'everyone survives' model would score {100 - pct_died:.1f}% accuracy.")
print("Which is exactly why we don't use accuracy.")
print()

if 10 <= pct_died <= 25:
    print(f"{pct_died:.1f}% mortality is plausible for ICU data.")
else:
    print(f"Note: {pct_died:.1f}% sits outside the 15-18% BOLD reports.")

An 'everyone survives' model would score 84.8% accuracy.
Which is exactly why we don't use accuracy.

15.2% mortality is plausible for ICU data.


## Step 3: Pick the predictors

For every column, ask one question: would a clinician know this at the bedside, at the
moment of the reading?

Yes, keep it (labs, vitals, blood gas, past SOFA scores, demographics). No, because
it's only known after the outcome, exclude it (future SOFA scores, length of stay,
timestamps, IDs, the outcome itself). Race/ethnicity is set aside too, not used as a
predictor, kept for a fairness check later.

Every column gets checked, nothing slips through unlabelled, and a final check
confirms nothing leaky made it into the predictor list.

In [9]:
# Match predictor groups by column prefix.

# Full blood count, clotting, metabolic panel, liver, misc enzymes.
lab_cols = [c for c in df.columns
            if c.startswith(("cbc_", "coag_", "bmp_", "hfp_", "others_"))]

# Bedside observations, plus the two oxygen saturation readings.
vital_cols = [c for c in df.columns if c.startswith("vitals_")]
vital_cols += [c for c in ["SpO2", "SaO2"] if c in df.columns]

gas_cols = [c for c in ["pH", "pCO2", "pO2", "Carboxyhemoglobin", "Methemoglobin"]
            if c in df.columns]

# Severity over the previous 24 hours.
severity_cols = [c for c in df.columns if c.startswith("sofa_past_")]

# Demographics and body measurements, all taken at admission.
demo_cols = [c for c in ["admission_age", "sex_female", "comorbidity_score_value",
                         "weight_admission", "height_admission", "BMI_admission"]
             if c in df.columns]

for name, cols in [("Labs", lab_cols), ("Vitals", vital_cols), ("Blood gas", gas_cols),
                   ("Severity (SOFA)", severity_cols), ("Demographics", demo_cols)]:
    print(f"{name} ({len(cols)}):")
    print(f"  {', '.join(cols)}")
    print()

Labs (32):
  cbc_wbc, cbc_hemoglobin, cbc_hematocrit, cbc_platelet, cbc_mch, cbc_mchc, cbc_mcv, cbc_rbc, cbc_rdw, coag_fibrinogen, coag_pt, coag_inr, coag_ptt, bmp_sodium, bmp_potassium, bmp_chloride, bmp_bicarbonate, bmp_bun, bmp_creatinine, bmp_glucose, bmp_aniongap, bmp_calcium, bmp_lactate, hfp_alt, hfp_alp, hfp_ast, hfp_bilirubin_total, hfp_bilirubin_direct, hfp_albumin, others_ck_cpk, others_ck_mb, others_ld_ldh

Vitals (8):
  vitals_heart_rate, vitals_resp_rate, vitals_mbp_ni, vitals_sbp_ni, vitals_dbp_ni, vitals_tempc, SpO2, SaO2

Blood gas (5):
  pH, pCO2, pO2, Carboxyhemoglobin, Methemoglobin

Severity (SOFA) (6):
  sofa_past_coagulation_24hr, sofa_past_liver_24hr, sofa_past_cardiovascular_24hr, sofa_past_cns_24hr, sofa_past_renal_24hr, sofa_past_overall_24hr

Demographics (6):
  admission_age, sex_female, comorbidity_score_value, weight_admission, height_admission, BMI_admission



In [ ]:
# Columns the model must not see, keyed by reason.
exclusions = {}

# Measured after the index reading.
exclusions["future SOFA scores (measured after the reading)"] = [
    c for c in df.columns if c.startswith("sofa_future_")
]

# Total LOS is only known once the admission ends.
exclusions["length of stay (only known once the admission ends)"] = [
    c for c in ["los_hospital", "los_ICU"] if c in df.columns
]

# Anything recorded at or after discharge.
exclusions["timestamps and discharge fields (at or after the outcome)"] = [
    c for c in df.columns
    if "_timestamp" in c or "datetime_" in c or "discharge" in c.lower()
]

# Identifiers carry no clinical signal.
exclusions["IDs and source database (no clinical meaning)"] = [
    c for c in ["unique_subject_id", "unique_hospital_admission_id",
                "unique_icustay_id", "source_db"] if c in df.columns
]

# The answer, plus race_ethnicity, kept aside for a later check.
exclusions["the outcome, and race_ethnicity (kept aside for a later check)"] = [
    c for c in [LABEL, "race_ethnicity"] if c in df.columns
]

for reason, cols in exclusions.items():
    if cols:
        print(f"EXCLUDED: {reason}")
        print(f"  {', '.join(cols)}")
        print()

In [11]:
FEATURES = lab_cols + vital_cols + gas_cols + severity_cols + demo_cols

# Check every column was either kept or excluded.
all_excluded = {c for cols in exclusions.values() for c in cols}
unaccounted = [c for c in df.columns if c not in FEATURES and c not in all_excluded]

print(f"Columns in data: {len(df.columns)}")
print(f"Kept:            {len(FEATURES)}")
print(f"Excluded:        {len(all_excluded)}")
print(f"Unaccounted:     {len(unaccounted)}")
print()

if unaccounted:
    print("WARNING, neither kept nor excluded:", ", ".join(unaccounted))
else:
    print("Every column accounted for.")

# Stop if anything leaky got through.
leaky = [c for c in FEATURES
         if any(p in c for p in ("sofa_future_", "los_", "_timestamp", "datetime_", "discharge"))]
assert not leaky, f"LEAKAGE, these must not be predictors: {leaky}"
assert LABEL not in FEATURES, "LEAKAGE, the outcome is in the feature list"
print("Leakage check passed.")

Columns in data: 71
Kept:            57
Excluded:        14
Unaccounted:     0

Every column accounted for.
Leakage check passed.


## Step 4: Split the data

Split patients into a training set (80%) and a test set (20%). The model learns from
training, then gets scored on test data it has never seen. Stratified means both sets
keep the same death rate, since deaths are rare and a plain random split could load
them unevenly.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURES]
y = df[LABEL]

# random_state fixes the split so results are reproducible. stratify=y keeps
# the death rate the same in both halves.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {len(X_train):,} patients, {y_train.mean() * 100:.1f}% died")
print(f"Test:  {len(X_test):,} patients, {y_test.mean() * 100:.1f}% died")

## Step 5: Train Random Forest

Some columns still have gaps (Step 1 showed this), and a Random Forest can't train on
gaps. Fill them with the median value of each column, the simplest defensible choice.
The median is learned from the training set only and reused on the test set, so the
test set stays something the model has genuinely never seen.

Then train a Random Forest. Default settings, since it works well without much tuning.
`class_weight="balanced"` tells it deaths matter more, since they're rare in the data.

In [ ]:
from sklearn.impute import SimpleImputer

# Learn the median from training data only, then apply it to both sets.
imputer = SimpleImputer(strategy="median")
X_train_filled = pd.DataFrame(
    imputer.fit_transform(X_train), columns=FEATURES, index=X_train.index
)
X_test_filled = pd.DataFrame(
    imputer.transform(X_test), columns=FEATURES, index=X_test.index
)

print(f"Gaps in training set before fill: {X_train.isna().sum().sum()}")
print(f"Gaps in training set after fill:  {X_train_filled.isna().sum().sum()}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# class_weight="balanced" tells the model deaths matter more, since they're rare.
rf = RandomForestClassifier(class_weight="balanced", random_state=42)
rf.fit(X_train_filled, y_train)

print(f"Random Forest trained on {len(X_train_filled):,} patients, {len(FEATURES)} features.")